# Evaluation Results Viewer

Load and explore evaluation results from `results/all_results.jsonl`

In [1]:
import pandas as pd
import json
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [2]:
RESULTS_FILE = Path("/home/hltcoe/rjha/rjha_exp/pylate-xtr/results/all_results.jsonl")

def load_results(filepath: Path = RESULTS_FILE) -> pd.DataFrame:
    """Load JSONL results and flatten nested structures."""
    records = []
    with open(filepath, 'r') as f:
        for line in f:
            data = json.loads(line)
            
            # Flatten all fields
            flat = {
                'dataset': data.get('dataset'),
                'model': data.get('model'),
                'model_dtype': data.get('model_dtype'),
                'embedding_dtype': data.get('embedding_dtype'),
                'query_length': data.get('query_length'),
                'doc_length': data.get('doc_length'),
                'lowercase': data.get('lowercase'),
                'k': data.get('k'),
                'k_token': data.get('k_token'),
                'retrieval_mode': data.get('retrieval_mode'),
                'index_time': data.get('index_time'),
                'retrieve_time': data.get('retrieve_time'),
                'encode_batch_size': data.get('encode_batch_size'),
                'retrieval_batch_size': data.get('retrieval_batch_size'),
                'timestamp': data.get('timestamp'),
                'run_id': data.get('run_id'),
            }
            
            # Add evaluation scores
            scores = data.get('evaluation_scores', {})
            flat['map'] = scores.get('map')
            flat['ndcg@10'] = scores.get('ndcg@10')
            flat['ndcg@100'] = scores.get('ndcg@100')
            flat['recall@10'] = scores.get('recall@10')
            flat['recall@100'] = scores.get('recall@100')
            flat['hit_rate@5'] = scores.get('hit_rate@5')
            
            # Add imputation info
            imputation = data.get('imputation', {})
            flat['imputation_method'] = imputation.get('method') if imputation else "min"
            flat['imputation_percentile'] = imputation.get('percentile') if imputation else None
            flat['power_law_multiplier'] = imputation.get('power_law_multiplier') if imputation else None
            
            # Add index config info
            idx_cfg = data.get('index_config', {})
            flat['index_type'] = idx_cfg.get('name')
            
            records.append(flat)
    
    df = pd.DataFrame(records)
    
    # Apply default values for missing columns (only for non-None defaults)
    defaults = {
        'retrieval_mode': 'ColBERT',  # Older runs were ColBERT by default
    }
    for col, default in defaults.items():
        if col in df.columns:
            df[col] = df[col].fillna(default)
    
    return df

df = load_results()
print(f"Loaded {len(df)} results")

Loaded 1762 results


In [3]:
def parse_dataset_name(dataset: str) -> str:
    """Extract readable dataset name from full path."""
    parts = dataset.split('/')
    if parts[0] == 'beir':
        # beir/nfcorpus/test -> nfcorpus, beir/trec-covid -> trec-covid
        return parts[1]
    elif parts[0] == 'lotte':
        # lotte/lifestyle/dev/search -> lotte-lifestyle
        return f"lotte-{parts[1]}"
    elif parts[0] == 'nano-beir':
        # nano-beir/msmarco -> nano-msmarco
        return f"nano-{parts[1]}"
    else:
        return dataset

def parse_model_name(model: str) -> str:
    """Extract model name, treating checkpoint-X and final as the same model."""
    p = Path(model)
    # Both 'checkpoint-XXXXX' and 'final' are subdirs of the actual model dir
    if 'checkpoint' in p.name or p.name == 'final':
        return p.parent.name
    return p.name

# Extract short names for readability
df['model_short'] = df['model'].apply(parse_model_name)
df['dataset_short'] = df['dataset'].apply(parse_dataset_name)

In [ ]:
# Default filters - used when not specified in filter()
DEFAULT_MODELS = [
    'experiment_1_contrastive_colbert_bs196_50k',
    'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k',
    'experiment_1_contrastive_xtr_primeqa_kprime_64_bs196_50k_bugfix',
    'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix',
    'experiment_1_contrastive_xtr_primeqa_kprime_256_bs196_50k_bugfix',
    'experiment_1_contrastive_xtr_primeqa_kprime_512_bs196_50k_bugfix',
    'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_10k_lr2e-4_bugfix',
    'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_lr2e-4_bugfix'
]

def filter_df(
    models: list | None = DEFAULT_MODELS,
    retrieval_modes: list | None = None,
    imputation_methods: list | None = None,
    datasets: list | None = None,
    k_tokens: list | None = None,
) -> pd.DataFrame:
    """Filter dataframe with defaults. Pass None to include all for that dimension."""
    result = df.copy()
    if models is not None:
        result = result[result['model_short'].isin(models)]
    if retrieval_modes is not None:
        result = result[result['retrieval_mode'].isin(retrieval_modes)]
    if imputation_methods is not None:
        result = result[result['imputation_method'].isin(imputation_methods)]
    if datasets is not None:
        result = result[result['dataset_short'].isin(datasets)]
    if k_tokens is not None:
        result = result[result['k_token'].isin(k_tokens)]
    return result

print(f"Total results: {len(df)}")
print(f"With default filter: {len(filter_df())}")

Total results: 1762
With default filter: 207


## Overview

In [28]:
fdf = filter_df()
print("Unique models:")
for m in fdf['model_short'].unique():
    print(f"  - {m}")
print(f"\nUnique datasets: {list(fdf['dataset_short'].unique())}")
print(f"Index types: {list(fdf['index_type'].unique())}")
print(f"Retrieval modes: {list(fdf['retrieval_mode'].unique())}")
print(f"Imputation methods: {list(fdf['imputation_method'].dropna().unique())}")

Unique models:
  - experiment_1_contrastive_colbert_bs196_50k
  - experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k
  - experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix
  - experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_lr2e-4_bugfix
  - experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_10k_lr2e-4_bugfix

Unique datasets: ['nfcorpus', 'fiqa', 'scidocs', 'scifact', 'trec-covid', 'nq', 'webis-touche2020', 'quora', 'lotte-lifestyle']
Index types: ['ScaNN']
Retrieval modes: ['ColBERT', 'XTR']
Imputation methods: ['min', 'percentile', 'mean', 'power_law', 'zero']


## Results Table

Key metrics formatted for readability (scores as percentages)

In [29]:
# Display key columns with formatted scores
fdf = filter_df(imputation_methods=['min'])  # Default to min imputation for clean comparison

display_cols = ['dataset_short', 'model_short', 'retrieval_mode', 'imputation_method', 
                'index_type', 'ndcg@10', 'ndcg@100', 'recall@100', 'map']
results_display = fdf[display_cols].copy()

# Format numeric columns as percentages
score_cols = ['ndcg@10', 'ndcg@100', 'recall@100', 'map']
for col in score_cols:
    results_display[col] = results_display[col].apply(lambda x: f"{x*100:.2f}" if pd.notna(x) else "")

results_display.sort_values(['dataset_short', 'model_short', 'retrieval_mode'])

,dataset_short,model_short,retrieval_mode,imputation_method,index_type,ndcg@10,ndcg@100,recall@100,map
1549,fiqa,experiment_1_contrastive_colbert_bs196_50k,ColBERT,min,ScaNN,37.15,43.02,65.60,31.25
1616,fiqa,experiment_1_contrastive_colbert_bs196_50k,XTR,min,ScaNN,36.84,42.60,65.40,30.73
1667,fiqa,experiment_1_contrastive_colbert_bs196_50k,XTR,min,ScaNN,36.67,42.62,65.92,30.63
1703,fiqa,experiment_1_contrastive_colbert_bs196_50k,XTR,min,ScaNN,36.69,42.73,65.76,30.79
1760,fiqa,experiment_1_contrastive_colbert_bs196_50k,XTR,min,ScaNN,35.87,41.85,64.50,30.14
...,...,...,...,...,...,...,...,...,...
1756,trec-covid,experiment_1_contrastive_xtr_primeqa_kprime_12...,XTR,min,ScaNN,77.70,56.42,13.48,10.54
1746,trec-covid,experiment_1_contrastive_xtr_primeqa_kprime_12...,XTR,min,ScaNN,67.80,48.00,11.30,8.14
1618,webis-touche2020,experiment_1_contrastive_colbert_bs196_50k,ColBERT,min,ScaNN,23.43,35.47,47.11,15.24
1619,webis-touche2020,experiment_1_contrastive_colbert_bs196_50k,ColBERT,min,ScaNN,23.43,35.47,47.11,15.24


In [30]:
fdf.model_short.unique()

array(['experiment_1_contrastive_colbert_bs196_50k',
       'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k',
       'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix',
       'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_lr2e-4_bugfix',
       'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_10k_lr2e-4_bugfix'],
      dtype=object)

In [45]:
filter_df(models=['experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix'], datasets=['nfcorpus'], imputation_methods=['min'], k_tokens=[40_000])

,dataset,model,model_dtype,embedding_dtype,query_length,doc_length,lowercase,k,k_token,retrieval_mode,index_time,retrieve_time,encode_batch_size,retrieval_batch_size,timestamp,run_id,map,ndcg@10,ndcg@100,recall@10,recall@100,hit_rate@5,imputation_method,imputation_percentile,power_law_multiplier,index_type,model_short,dataset_short
1631,beir/nfcorpus/test,output/experiment_1_contrastive_xtr_primeqa_kp...,fp32,fp32,32.0,300.0,False,100,40000,XTR,34.126966,136.280786,2000.0,32.0,2026-02-02T16:30:06.736492,ecbe7333ca,0.152379,0.330829,0.297247,0.161561,0.298804,0.631579,min,10.0,100.0,ScaNN,experiment_1_contrastive_xtr_primeqa_kprime_12...,nfcorpus
1676,beir/nfcorpus/test,output/experiment_1_contrastive_xtr_primeqa_kp...,fp32,fp32,32.0,300.0,False,100,40000,XTR,0.000000,220.441559,1000.0,32.0,2026-02-03T14:40:15.866774,ecbe7333ca,0.152362,0.330710,0.297234,0.161494,0.298804,0.631579,min,10.0,100.0,ScaNN,experiment_1_contrastive_xtr_primeqa_kprime_12...,nfcorpus
1761,beir/nfcorpus/test,output/experiment_1_contrastive_xtr_primeqa_kp...,fp16,fp16,32.0,300.0,False,100,40000,XTR,0.000000,225.173374,1000.0,32.0,2026-02-04T16:49:13.339795,7f861543d9,0.153729,0.332537,0.297731,0.165009,0.297579,0.622291,min,10.0,100.0,ScaNN,experiment_1_contrastive_xtr_primeqa_kprime_12...,nfcorpus


## Pivot Table: NDCG@10 by Dataset

In [31]:
# Pivot table: NDCG@10 by dataset and model
fdf = filter_df(imputation_methods=['min'])

pivot = fdf.pivot_table(
    index='dataset_short', 
    columns=['model_short', 'retrieval_mode'], 
    values='ndcg@10',
    aggfunc='max'
) * 100

# Create a mapping of column names to indices
col_mapping = {col: i for i, col in enumerate(pivot.columns)}
legend_df = pd.DataFrame(list(col_mapping.items()), columns=['Model Configuration', 'Index'])

# Rename columns to indices
pivot_indexed = pivot.copy()
pivot_indexed.columns = [col_mapping[col] for col in pivot.columns]

print("Column Legend:")
display(legend_df.style.set_properties(subset=['Model Configuration'], **{'width': '500px', 'text-align': 'left'}))
print("\nResults:")

pivot_indexed.round(2).style.background_gradient(cmap='YlGn', axis="columns").format("{:.2f}").set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'right')]}
])

Column Legend:


,Model Configuration,Index
0,"('experiment_1_contrastive_colbert_bs196_50k', 'ColBERT')",0
1,"('experiment_1_contrastive_colbert_bs196_50k', 'XTR')",1
2,"('experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_10k_lr2e-4_bugfix', 'XTR')",2
3,"('experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k', 'XTR')",3
4,"('experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix', 'XTR')",4
5,"('experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_lr2e-4_bugfix', 'XTR')",5



Results:


,0,1,2,3,4,5
dataset_short,,,,,,
fiqa,37.15,36.84,30.29,36.87,34.18,27.45
lotte-lifestyle,49.32,49.30,nan,nan,46.71,nan
nfcorpus,34.49,34.38,32.55,35.00,33.25,31.88
nq,53.07,52.80,nan,51.72,nan,nan
quora,87.50,87.41,nan,nan,nan,nan
scidocs,18.54,18.54,15.25,17.07,16.94,13.06
scifact,70.59,69.88,62.55,66.18,64.51,60.07
trec-covid,78.45,78.38,76.05,78.93,78.14,67.80
webis-touche2020,23.43,23.20,nan,nan,nan,nan


## Compare Retrieval Modes

In [ ]:
# Compare ColBERT vs XTR retrieval modes
fdf = filter_df(imputation_methods=['min'])

mode_comparison = fdf.pivot_table(
    index=['dataset_short', 'model_short'],
    columns='retrieval_mode',
    values='ndcg@10',
    aggfunc='max'
) * 100

mode_comparison.round(2)

## Compare Imputation Methods (XTR)

In [ ]:
# Compare imputation methods across all datasets for XTR model
XTR_MODEL = 'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix'
COLBERT_MODEL = 'experiment_1_contrastive_colbert_bs196_50k'

xtr_df = filter_df(
    models=[XTR_MODEL],
    retrieval_modes=['XTR'],
    datasets=['nfcorpus', 'fiqa', 'trec-covid', 'nq'],
    k_tokens=[10_000, 40_000],
)

if len(xtr_df) > 0:
    imputation_pivot = xtr_df.pivot_table(
        index=['dataset_short', 'k_token'],
        columns=['imputation_method'],
        values='ndcg@10',
        aggfunc='max'
    ) * 100
    
    styled = imputation_pivot.round(2).style.format("{:.2f}").set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'right')]}
    ])
    display(styled)
else:
    print("No XTR runs found for this model")